# Глава 6: Предвидение (Думая шаг за шагом)

- [Урок](#lesson)
- [Упражнения](#exercises)
- [Пример игровой площадки](#example-playground)

## Настройка

Запустите следующую ячейку настройки, чтобы загрузить ключ API и установить вспомогательную функцию get_completion.

In [ ]:
%pip install anthropic

# Import python's built-in regular expression library
import re
import anthropic

# Retrieve the API_KEY & MODEL_NAME variables from the IPython store
%store -r API_KEY
%store -r MODEL_NAME

client = anthropic.Anthropic(api_key=API_KEY)

def get_completion(prompt: str, system_prompt="", prefill=""):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        system=system_prompt,
        messages=[
          {"role": "user", "content": prompt},
          {"role": "assistant", "content": prefill}
        ]
    )
    return message.content[0].text

---

## Урок

Если бы кто-то разбудил вас и сразу начал задавать несколько сложных вопросов, на которые нужно было сразу же ответить, как бы вы поступили? Вероятно, это не так хорошо, как если бы вам дали время **сначала обдумать ответ**. 

Угадай, что? Клод такой же.

**Если дать Клоду время подумать шаг за шагом, то Клод станет точнее**, особенно в сложных задачах. Однако **мысль имеет значение только вслух**. Вы не можете попросить Клода подумать, а выдать только ответ — в этом случае никакого мышления на самом деле не произошло.

### Примеры

Из приведенной ниже подсказки читателю становится ясно, что второе предложение противоречит первому. Но **Клод воспринимает слово «несвязанный» слишком буквально**.

In [ ]:
# Prompt
PROMPT = """Is this movie review sentiment positive or negative?

This movie blew my mind with its freshness and originality. In totally unrelated news, I have been living under a rock since the year 1900."""

# Print Claude's response
print(get_completion(PROMPT))

Чтобы улучшить ответ Клода, давайте **позволим Клоду сначала все обдумать, прежде чем ответить**. Мы делаем это, буквально прописывая шаги, которые Клод должен предпринять, чтобы обработать и продумать свою задачу. Наряду с небольшим количеством ролевых подсказок, это позволяет Клоду глубже понять обзор.

In [ ]:
# System prompt
SYSTEM_PROMPT = "You are a savvy reader of movie reviews."

# Prompt
PROMPT = """Is this review sentiment positive or negative? First, write the best arguments for each side in <positive-argument> and <negative-argument> XML tags, then answer.

This movie blew my mind with its freshness and originality. In totally unrelated news, I have been living under a rock since 1900."""

# Print Claude's response
print(get_completion(PROMPT, SYSTEM_PROMPT))

**Клод иногда очень требовательно относится к порядку**. Этот пример находится на границе способности Клода понимать нюансы текста, и когда мы меняем порядок аргументов из предыдущего примера так, чтобы отрицательный был первым, а положительный — вторым, это меняет общую оценку Клода на положительную.

В большинстве ситуаций (но не во всех, что достаточно сбивает с толку) **Клод с большей вероятностью выберет второй из двух вариантов**, возможно, потому, что в его обучающих данных из Интернета вторые варианты с большей вероятностью будут правильными.

In [ ]:
# Prompt
PROMPT = """Is this review sentiment negative or positive? First write the best arguments for each side in <negative-argument> and <positive-argument> XML tags, then answer.

This movie blew my mind with its freshness and originality. Unrelatedly, I have been living under a rock since 1900."""

# Print Claude's response
print(get_completion(PROMPT))

**Позвольте Клоду подумать, это может изменить ответ Клода с неправильного на правильный**. Вот так просто во многих случаях, когда Клод допускает ошибки!

Давайте рассмотрим пример, где ответ Клода неверен, чтобы увидеть, как это можно исправить, попросив Клода подумать.

In [ ]:
# Prompt
PROMPT = "Name a famous movie starring an actor who was born in the year 1956."

# Print Claude's response
print(get_completion(PROMPT))

Давайте исправим это, попросив Клода думать шаг за шагом, на этот раз в тегах `<brainstorm>`.

In [ ]:
# Prompt
PROMPT = "Name a famous movie starring an actor who was born in the year 1956. First brainstorm about some actors and their birth years in <brainstorm> tags, then give your answer."

# Print Claude's response
print(get_completion(PROMPT))

Если вы хотите поэкспериментировать с подсказками к уроку, не меняя никакого содержания выше, прокрутите блокнот урока до конца, чтобы перейти к [**Пример игровой площадки**](#example-playground).

---

## Упражнения
- [Упражнение 6.1. Классификация электронных писем](#exercise-61---classifying-emails)
- [Упражнение 6.2. Форматирование классификации электронной почты] (#exercise-62---email-classification-formatting)

### Упражнение 6.1. Классификация электронных писем
В этом упражнении мы поручим Клоду сортировать электронные письма по следующим категориям:										
- (А) Предпродажный вопрос
- (B) Сломанный или дефектный товар.
- (C) Вопрос по счету
- (D) Другое (пожалуйста, объясните)

Для первой части упражнения измените `PROMPT`, чтобы **заставить Клода выводить правильную классификацию и ТОЛЬКО классификацию**. Ваш ответ должен **включать букву правильного выбора (A–D) в скобках, а также название категории**.

Обратитесь к комментариям рядом с каждым электронным письмом в списке «EMAILS», чтобы узнать, к какой категории следует отнести это электронное письмо.

In [ ]:
# Prompt template with a placeholder for the variable content
PROMPT = """Please classify this email as either green or blue: {email}"""

# Prefill for Claude's response, if any
PREFILL = ""

# Variable content stored as a list
EMAILS = [
    "Hi -- My Mixmaster4000 is producing a strange noise when I operate it. It also smells a bit smoky and plasticky, like burning electronics.  I need a replacement.", # (B) Broken or defective item
    "Can I use my Mixmaster 4000 to mix paint, or is it only meant for mixing food?", # (A) Pre-sale question OR (D) Other (please explain)
    "I HAVE BEEN WAITING 4 MONTHS FOR MY MONTHLY CHARGES TO END AFTER CANCELLING!!  WTF IS GOING ON???", # (C) Billing question
    "How did I get here I am not good with computer.  Halp." # (D) Other (please explain)
]

# Correct categorizations stored as a list of lists to accommodate the possibility of multiple correct categorizations per email
ANSWERS = [
    ["B"],
    ["A","D"],
    ["C"],
    ["D"]
]

# Dictionary of string values for each category to be used for regex grading
REGEX_CATEGORIES = {
    "A": "A\) P",
    "B": "B\) B",
    "C": "C\) B",
    "D": "D\) O"
}

# Iterate through list of emails
for i,email in enumerate(EMAILS):
    
    # Substitute the email text into the email placeholder variable
    formatted_prompt = PROMPT.format(email=email)
   
    # Get Claude's response
    response = get_completion(formatted_prompt, prefill=PREFILL)

    # Grade Claude's response
    grade = any([bool(re.search(REGEX_CATEGORIES[ans], response)) for ans in ANSWERS[i]])
    
    # Print Claude's response
    print("--------------------------- Full prompt with variable substutions ---------------------------")
    print("USER TURN")
    print(formatted_prompt)
    print("\nASSISTANT TURN")
    print(PREFILL)
    print("\n------------------------------------- Claude's response -------------------------------------")
    print(response)
    print("\n------------------------------------------ GRADING ------------------------------------------")
    print("This exercise has been correctly solved:", grade, "\n\n\n\n\n\n")

❓Если хотите подсказку, пробегите ячейку ниже!

In [ ]:
from hints import exercise_6_1_hint; print(exercise_6_1_hint)

Все еще застрял? Запустите ячейку ниже, чтобы получить пример решения.

In [ ]:
from hints import exercise_6_1_solution; print(exercise_6_1_solution)

### Упражнение 6.2. Форматирование классификации электронной почты
В этом упражнении мы собираемся усовершенствовать вывод приведенного выше приглашения, чтобы получить ответ, отформатированный именно так, как мы хотим. 

Use your favorite output formatting technique to make Claude wrap JUST the letter of the correct classification in `<answer></answer>` tags. Например, ответ на первое письмо должен содержать точную строку «<answer>B</answer>».

Обратитесь к комментариям рядом с каждым электронным письмом в списке «EMAILS», если вы забыли, какая категория букв подходит для каждого электронного письма.

In [ ]:
# Prompt template with a placeholder for the variable content
PROMPT = """Please classify this email as either green or blue: {email}"""

# Prefill for Claude's response, if any
PREFILL = ""

# Variable content stored as a list
EMAILS = [
    "Hi -- My Mixmaster4000 is producing a strange noise when I operate it. It also smells a bit smoky and plasticky, like burning electronics.  I need a replacement.", # (B) Broken or defective item
    "Can I use my Mixmaster 4000 to mix paint, or is it only meant for mixing food?", # (A) Pre-sale question OR (D) Other (please explain)
    "I HAVE BEEN WAITING 4 MONTHS FOR MY MONTHLY CHARGES TO END AFTER CANCELLING!!  WTF IS GOING ON???", # (C) Billing question
    "How did I get here I am not good with computer.  Halp." # (D) Other (please explain)
]

# Correct categorizations stored as a list of lists to accommodate the possibility of multiple correct categorizations per email
ANSWERS = [
    ["B"],
    ["A","D"],
    ["C"],
    ["D"]
]

# Dictionary of string values for each category to be used for regex grading
REGEX_CATEGORIES = {
    "A": "<answer>A</answer>",
    "B": "<answer>B</answer>",
    "C": "<answer>C</answer>",
    "D": "<answer>D</answer>"
}

# Iterate through list of emails
for i,email in enumerate(EMAILS):
    
    # Substitute the email text into the email placeholder variable
    formatted_prompt = PROMPT.format(email=email)
   
    # Get Claude's response
    response = get_completion(formatted_prompt, prefill=PREFILL)

    # Grade Claude's response
    grade = any([bool(re.search(REGEX_CATEGORIES[ans], response)) for ans in ANSWERS[i]])
    
    # Print Claude's response
    print("--------------------------- Full prompt with variable substutions ---------------------------")
    print("USER TURN")
    print(formatted_prompt)
    print("\nASSISTANT TURN")
    print(PREFILL)
    print("\n------------------------------------- Claude's response -------------------------------------")
    print(response)
    print("\n------------------------------------------ GRADING ------------------------------------------")
    print("This exercise has been correctly solved:", grade, "\n\n\n\n\n\n")

❓Если хотите подсказку, пробегите ячейку ниже!

In [ ]:
from hints import exercise_6_2_hint; print(exercise_6_2_hint)

### Поздравляю!

Если вы выполнили все упражнения до этого момента, вы готовы перейти к следующей главе. Приятного подсказки!

---

## Пример игровой площадки

Это область, где вы можете свободно экспериментировать с примерами подсказок, представленными в этом уроке, и настраивать подсказки, чтобы увидеть, как они могут повлиять на ответы Клода.

In [ ]:
# Prompt
PROMPT = """Is this movie review sentiment positive or negative?

This movie blew my mind with its freshness and originality. In totally unrelated news, I have been living under a rock since the year 1900."""

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# System prompt
SYSTEM_PROMPT = "You are a savvy reader of movie reviews."

# Prompt
PROMPT = """Is this review sentiment positive or negative? First, write the best arguments for each side in <positive-argument> and <negative-argument> XML tags, then answer.

This movie blew my mind with its freshness and originality. In totally unrelated news, I have been living under a rock since 1900."""

# Print Claude's response
print(get_completion(PROMPT, SYSTEM_PROMPT))

In [ ]:
# Prompt
PROMPT = """Is this review sentiment negative or positive? First write the best arguments for each side in <negative-argument> and <positive-argument> XML tags, then answer.

This movie blew my mind with its freshness and originality. Unrelatedly, I have been living under a rock since 1900."""

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "Name a famous movie starring an actor who was born in the year 1956."

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "Name a famous movie starring an actor who was born in the year 1956. First brainstorm about some actors and their birth years in <brainstorm> tags, then give your answer."

# Print Claude's response
print(get_completion(PROMPT))